# Decision Tree Development Notebook (v1) 

In [1]:
import numpy as np
import pandas as pd
from collections import Counter
from functools import reduce
from typing import Literal

In [ ]:
def prod(arr: list) -> float:
    """Computes the product of all elevents of a given vector"""
    return reduce(lambda x, y: x * y, arr, 1)

def gini(arr: list) -> float:
    """Computes Gini Impurity for a given label vector"""
    n = len(arr)
    value_counts = dict(Counter(arr))
    unique_categories = list(value_counts.keys())
    prob_list = {cat: (value_counts[cat] / n) for cat in unique_categories}
    return 1 - sum(p**2 for p in prob_list.values())

def get_cutoff(feature, y, base_gini, return_gain: bool = False):
    """Searches a good cutoff in a feature vector in order to maximize node purity gain"""
    feature_nrows_unique = len(set(feature))
    cuts = int(np.floor(np.min([feature_nrows_unique, 10])))
    candidates = list(set([np.quantile(feature, q=q).item() for q in np.linspace(0, 1, cuts)]))
    curr_winner, curr_winner_gain = None, 0
    for candidate in candidates:
        left_gini, right_gini = gini(y[feature < candidate]), gini(y[feature >= candidate])
        gain = (base_gini - left_gini) + (base_gini - right_gini)  # TODO: Generalize
        if gain > curr_winner_gain:
            curr_winner_gain, curr_winner = gain, candidate
    return curr_winner if return_gain is False else (curr_winner, curr_winner_gain)

def rates_per_category(y, cats: list[int]):
    """Retrieves the percentage of data that corresponds to each class/category"""
    return [(y == cat).mean().item() for cat in cats]


In [ ]:
# Root node -> Populate in function of a certain feature (index) -> Search 

class BinaryTree:
    """Manual implementation of a BinaryTree"""

    def __init__(self, gini: float = None, proba: list[float] = None, chained_conditions: list[tuple[int, str, float]] = None):
        # Data structure info
        self.left = None  # Leaf (or "< Node")
        self.right = None  # Leaf (or ">= Node")
        self.parent = None  # To comeback
        
        # Training state
        self.feature_index: int = None  # Index of the feature used to split current node
        self.cutoff_value: float = None  # Boundary
        
        self.gini: float = gini  # Impurity
        self.proba: list[float] = proba  # List of probabilities (not calibrated yet)
        self.chained_conditions: list[tuple[int, str, float]] = chained_conditions  # Necessary conditions to arrive to the node
        if chained_conditions is None:
            self.chained_conditions = []

        self.branch_status = None

    def _set_parent(self, parent) -> None:
        self.parent = parent
        if hasattr(self.parent, "chained_conditions"):
            if self.parent.chained_conditions is not None:
                self.chained_conditions = self.parent.chained_conditions + self.chained_conditions

    def _set_child(self, child, direction: Literal["left", "right"]) -> None:
        assert direction in ["left", "right"], "<direction> value must be either left or right"
        if direction == "left":
            self.left = child
        else:
            self.right = child
        child._set_parent(self)

    def _set_children(self, children: list) -> None:
        assert len(children) == 2, f"Number of children must be 2"
        child_left, child_right = children[0], children[1]
        self._set_child(child_left, "left")
        self._set_child(child_right, "right")

    def _set_branch_status(self, branch_status: Literal["OK"]) -> None:
        self.branch_status = branch_status

    def _is_leaf(self):
        return (self.left is None) and (self.right is None)

    def __repr__(self):
        str1 = "<BinaryTree object>"
        str2 = f"Left child = {type(self.left)}"
        str3 = f"Right child = {type(self.right)}"
        str4 = f"Feature index (on train data) = {self.feature_index}"
        str5 = f"Cutoff value in feature vector = {self.cutoff_value}"
        str6 = f"Impurity = {self.gini}"
        str7 = f"Mean label = {self.proba}"
        str8 = f"Parent node = {type(self.parent)}"
        str9 = f"Node conditions = {self.chained_conditions}"
        str10 = f"BRANCH STATUS - {self.branch_status}"
        final_str = "\n - ".join([str1, str2, str3, str4, str5, str6, str7, str8, str9, str10])
        return final_str

    @staticmethod
    def filter_data(data, queries: list[tuple[int, str, float]], reference_data):
        X = reference_data
        if len(queries) > 0:
            query_str = " & ".join([f"(X[:, {t[0]}] {t[1]} {t[2]})" for t in queries])
            str_to_eval = f"data[{query_str}]"
            return eval(str_to_eval)
        return data

In [212]:
class DecisionTreeClassifier:
    """Manual implementation of DecisionTreeClassifier"""

    def __init__(self):
        self.tree = BinaryTree()
        self.trained = False

    def train_node(self, X, y) -> None:
        ...
    
    def train(self, X, y) -> None:
        """
        Executes the training process, note that this is an inplace logic
        Reminder: stop criteria is gini_of_node = 0
        """
        if self.trained is True:
            raise ValueError("Tree is already trained")

        # If Pandas...
        if hasattr(X, "values"):
            X = X.values
        if hasattr(y, "values"):
            y = y.values

        n_features = X.shape[1]
        cats = sorted(list(set(y)))

        # Start from root
        subtree = self.tree
        while True:
            print(id(subtree))
            if (subtree.left is not None) and (subtree.right is not None):
                if (subtree.left.branch_status == "OK") and (subtree.right.branch_status == "OK"):
                    subtree._set_branch_status("OK")
                    continue
            
            X_subtree = self.tree.filter_data(X, queries=subtree.chained_conditions, reference_data=X)
            y_subtree = self.tree.filter_data(y, queries=subtree.chained_conditions, reference_data=X)
            subtree_impurity = gini(y_subtree)
            
            purity_gains = []
            for feature_index in range(n_features):
                cutoff_value, purity_gain = get_cutoff(X_subtree[:, feature_index], y_subtree, base_gini=subtree_impurity, return_gain=True)
                purity_gains.append((feature_index, cutoff_value, purity_gain))
            best_feature_index, best_cutoff_value, _ = max(purity_gains, key=lambda t: t[2])

            # Set split info
            subtree.feature_index = best_feature_index
            subtree.cutoff_value = best_cutoff_value
            subtree.gini = subtree_impurity
            subtree.proba = rates_per_category(y_subtree, cats)

            # Feature space subset
            y_left = y_subtree[X_subtree[:, best_feature_index] < best_cutoff_value]
            y_right = y_subtree[X_subtree[:, best_feature_index] >= best_cutoff_value]

            child_left = BinaryTree(
                gini=gini(y_left),
                proba=rates_per_category(y_left, cats),
                chained_conditions=[(best_feature_index, "<", best_cutoff_value)],
            )
            child_right = BinaryTree(
                gini=gini(y_right),
                proba=rates_per_category(y_right, cats),
                chained_conditions=[(best_feature_index, ">=", best_cutoff_value)]
            )
            subtree._set_children([child_left, child_right])

            # Small enough
            if child_left.gini < 0.125:
                child_left._set_branch_status("OK")
            if child_right.gini < 0.125:
                child_right._set_branch_status("OK")

            if (subtree.left.branch_status == "OK") and (subtree.right.branch_status == "OK"):
                subtree._set_branch_status("OK")
                subtree = subtree.parent
                continue
            
            if subtree.left.branch_status != "OK":
                subtree = subtree.left
            elif subtree.right.branch_status != "OK":
                subtree = subtree.right
            else:
                subtree = subtree.parent
                subtree._set_branch_status("OK")
                if (subtree == self.tree) and (subtree.left.branch_status == "OK") and (subtree.right.branch_status == "OK"):
                    break
        self.trained = True

    # def _single_inference(self, X) -> float:
    #     raise NotImplementedError

    # def inference(self, X) -> float:
    #     raise NotImplementedError


In [213]:
dtc = DecisionTreeClassifier()
data = pd.read_parquet("../example_data/iris_dataset.parquet")
# data
dtc.train(data.drop(columns=["target"]).values, data["target"].values)

2840554858096
2840554858416
2840554859216
2840554860016
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
2840554859216
284055

KeyboardInterrupt: 

In [206]:
dtc.tree.right

<BinaryTree object>
 - Left child = <class '__main__.BinaryTree'>
 - Right child = <class '__main__.BinaryTree'>
 - Feature index (on train data) = 3
 - Cutoff value in feature vector = 1.8
 - Impurity = 0.5
 - Mean label = [0.0, 0.5, 0.5]
 - Parent node = <class '__main__.BinaryTree'>
 - Node conditions = [(2, '>=', 2.6333333333333306)]
 - BRANCH STATUS - None

In [16]:
id(dtc.tree.left)

2840417619472

In [13]:
dtc.tree.left

<BinaryTree object>
 - Left child = <class 'NoneType'>
 - Right child = <class 'NoneType'>
 - Feature index (on train data) = None
 - Cutoff value in feature vector = None
 - Impurity = None
 - Mean label = [1.0, 0.0, 0.0]
 - Parent node = <class '__main__.BinaryTree'>

In [68]:
dtc.tree.left

<BinaryTree object>
- Left child = <class 'NoneType'>
- Right child = <class 'NoneType'>
- Feature index (on train data) = None
- Cutoff value in feature vector = None
- Impurity = None
- Mean label = [1.0, 0.0, 0.0]

In [70]:
dtc.tree.right

<BinaryTree object>
- Left child = <class 'NoneType'>
- Right child = <class 'NoneType'>
- Feature index (on train data) = None
- Cutoff value in feature vector = None
- Impurity = None
- Mean label = [0.0, 0.5, 0.5]

In [69]:
x = data.drop(columns=["target"]).iloc[0, :].values
x

array([5.1, 3.5, 1.4, 0.2])

In [37]:
tree

<BinaryTree object>
- Left child = None
- Right child = None
- Feature index (on train data) = None
- Cutoff value in feature vector = None
- Impurity = None

In [20]:
# def _single_inference(dtc, x):
tree = dtc.tree
feat_index, cutoff = tree.feature_index, tree.cutoff_value